In [ ]:
import os
import sys

from corner import corner

sys.path.append('..')

from src.simulator import Model, BurstSimulator
from src.flow_matching.simulator import Model as BatchedModel

from src.c2st import c2st
from src.flow_matching.loader import empty_classifier_from_config, empty_model_from_config, read_config, prob_path_from_config
from src.flow_matching.distributions import UniformPrior, CompositePrior, Posterior, DiscreteUniform
from src.flow_matching.probability_path import GuidedLinearProbabilityPath
from src.flow_matching.integration import EulerODESolver
from src.flow_matching.models import MLPGuidedVectorField, FRBLightCurveCNN, LightCurveThinner, fourier_embedding, LightCurveMLP, UNetEncoder, TransdimensionalModel, EncodedClassifier
from src.flow_matching.transformer import TransformerGuidedField
from src.helpers import record_every, plot_posterior_samples, gen_parameter_labels

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.lines as mlines

import numpy as np
import pandas as pd
import torch
import yaml

from src.flow_matching.plotting import plot_loss, plot_snapshots
from src.flow_matching.helpers import choose_device, build_mlp, find_run_dir

device = choose_device()

# Loading in the FM model

In [ ]:
# fill in desired job_id or directory name 
job_id = "14403966" #False
run_dir = None
save_dir = "../checkpoints/"

In [ ]:
# loading the model (via run_id, or path)
run_dir = find_run_dir(job_id, save_dir) if job_id else os.path.join(save_dir, run_dir)

checkpoint_path = os.path.join(run_dir, 'training_checkpoint.pth')
config_path     = os.path.join(run_dir, 'config.yaml')

In [ ]:
# create empty model from config
config = read_config(config_path)
vector_field = empty_model_from_config(config)

In [ ]:
transdimensional = not config['training']['fixed_N'] #False
print(transdimensional)
if transdimensional:
    classifier = empty_classifier_from_config(config)
    vector_field = TransdimensionalModel(classifier, vector_field)

In [ ]:
# load trained model 
print(checkpoint_path)
checkpoint = torch.load(checkpoint_path, weights_only=False)

# Load 'normal' or EMA version 
ema = False
if ema:
    EMA_checkpoint_path = os.path.join(run_dir, "EMA_checkpoint.pth")
    state_dict = torch.load(EMA_checkpoint_path, weights_only=False)
    vector_field.load_state_dict(state_dict) 
else:
    vector_field.load_state_dict(checkpoint["model_state_dict"])

vector_field.eval()
vector_field.to(device)

losses = checkpoint["losses"]

In [ ]:
path = prob_path_from_config(config)
inf_params = config['model']['init_params']['inf_params']
N = path.p_data.model_params['ncomp']
vector_dim = N * len(inf_params)
burstparams = path.p_data.model_params['burstparams']

try:
    mean, std = torch.tensor(config['training']['sample_mean'], device=device), torch.tensor(config['training']['sample_std'], device=device)
except KeyError:
    mean, std = torch.zeros(vector_dim, device=device), torch.ones(vector_dim, device=device)

path.p_data.N_prior.device = device
path.p_data.device = device
path.p_simple.device = device
path.p_simple.set_prior_device()

In [ ]:
# path.p_data.noise='gaussian'
print(path.p_data.noise)
plt.plot(path.p_data.sample(5)[1][2].cpu())

# generate FM posterior conditioned on **observed** counts

## Read observational data sample

In [ ]:
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+241.347_all_data.dat'
filepath = '../' 'observational_data/FRBs/FRB20190124F_lc.dat'
# filepath = '../' 'observational_data/pulsar/20191221A_original_data.h5'

if 'magnetar' in filepath:
    df = pd.read_csv(filepath, header=None, delimiter=' ', usecols=[1])
    observed_counts = torch.tensor(df[1], device=device) 
elif 'FRB' in filepath:
    df = pd.read_csv(filepath, header=None, delimiter=' ')
    observed_counts = torch.tensor(df[0], device=device) #* 5 + 5
else: # pulsar
    import h5py
    with h5py.File(filepath, "r") as f:
        # Print all root level object names (aka keys) 
        # these can be group or dataset names 
        print(f.keys())
        data = f.get('profile')
        waterfall = np.array(f.get('waterfall'))
        observed_counts = torch.tensor(data, device=device) #* 6 + 5
    # read file

# plt.plot(observed_counts.cpu())
plt.plot(observed_counts.cpu())
# observed_counts = observed_counts[::2]
observed_counts.shape

# plt.imshow(waterfall)

In [ ]:
# downsample pulsar
def downsample(profile, factor):
    for i in range(factor):
        bins_1 = profile[::2]
        bins_2 = profile[1::2]
        if len(bins_1) != len(bins_2):
            bins_1_new = torch.zeros_like(bins_2)
            bins_1_new = bins_1[:-1]
            bins_1 = bins_1_new
        profile = bins_1 + bins_2
    return profile 

downsampled = downsample(profile=observed_counts, factor=2)
plt.plot(downsampled.cpu())
downsampled.shape

def pad_with_noise(profile, noise_level, noise:str, new_length=1000):
    assert len(profile) < new_length, 'profile exceeds desired length, padding not possible.'
    assert noise in ['poisson', 'gaussian'], 'noise must be either gaussian or poisson.'

    noise_padding_size = new_length - len(profile)
    padding_left = noise_padding_size // 2 + noise_padding_size % 2
    padding_right = noise_padding_size // 2
    
    if noise == "poisson":
        noise_left = torch.poisson(torch.ones(padding_left, device=device) * noise_level)
        noise_right = torch.poisson(torch.ones(padding_right, device=device) * noise_level)

    elif noise == "gaussian": 
        noise_left = torch.randn(int(padding_left), device=device) * noise_level
        noise_right = torch.randn(int(padding_right), device=device) * noise_level

    prepped_counts = torch.zeros(new_length)
    prepped_counts[:padding_left] = noise_left
    prepped_counts[new_length - padding_right:] = noise_right
    prepped_counts[padding_left:new_length - padding_right] = profile 
    return prepped_counts

prepped = pad_with_noise(downsampled, noise_level=1, noise='gaussian', new_length=int(1.4*len(downsampled)))
plt.plot(prepped)

In [ ]:
# prep magnetar burst

# reduce time resolution dt 5e-4 -> 1e-3
bins_1 = observed_counts[::2]
bins_2 = observed_counts[1::2]
if len(bins_1) != len(bins_2):
    bins_1_new = torch.zeros_like(bins_2)
    bins_1_new = bins_1[:-1]
    bins_1 = bins_1_new
new_bins = bins_1 + bins_2
print(new_bins.shape)

noise_padding_size = 1000 - len(new_bins)
padding_left = noise_padding_size // 2 + noise_padding_size % 2
padding_right = noise_padding_size // 2

noise_left = torch.poisson(torch.ones(padding_left, device=device)*3)#*3
noise_right = torch.poisson(torch.ones(padding_right, device=device)*3)#*3

prepped_counts = torch.zeros(1000)
prepped_counts[:padding_left] = noise_left
prepped_counts[1000 - padding_right:] = noise_right
prepped_counts[padding_left:1000 - padding_right] = new_bins
observed_counts = prepped_counts - 2
plt.plot(prepped_counts)

In [ ]:
num_samples = 2500  # number of prior samples to transform 
samples_per_batch = 2500
batches = num_samples // samples_per_batch
final_snapshot = torch.zeros((batches * samples_per_batch, vector_dim), device=device)
Ns = torch.zeros((batches * samples_per_batch, 1), device=device)

# use same data point for conditioning all prior samples
# condition = torch.tensor((observed_counts - 3) / 150, device=device, dtype=torch.float) subtract bg for magnetars
condition = torch.tensor(observed_counts / 150, device=device, dtype=torch.float)
simulations = condition.repeat(samples_per_batch, 1)

# initialize ODE solver
solver = EulerODESolver(vector_field)
nts = 200
ts = torch.linspace(0, 1, nts).to(device)

transdimensional=True
if transdimensional:
    # logits = classifier(condition.unsqueeze(0))
    logits = classifier(condition.unsqueeze(0))
    p_N = torch.softmax(logits, dim=1).flatten()
else:
    p_N = torch.zeros(N, device=device)
    p_N[N-1] = 1
    
# integrate in batches
for i in range(batches):
    # simulate ODE starting from x0

    start = i * samples_per_batch
    stop = start + samples_per_batch

    N_samples = torch.multinomial(p_N.repeat(samples_per_batch, 1), num_samples=1) + 1

    x0 = path.p_simple.sample(samples_per_batch, Ns=N_samples).to(device)
    # # print(x0.shape)
    # print(x0)
    # x0 = (torch.randn(size=(samples_per_batch, vector_dim), device=device) )

    Ns[start:stop, :] = N_samples
    final_snapshot[start:stop, :] = solver.solve(x0, ts.view(1, nts, 1).expand(samples_per_batch, nts, 1), y=simulations, N=N_samples) * std + mean

In [ ]:
final_snapshot

In [ ]:
import arviz
print(final_snapshot.cpu().numpy()[None, :, :].shape)
var_names = gen_parameter_labels(inf_params, N)
arviz_samples = final_snapshot[(Ns == 3).view(Ns.size(0))]# samples where N == N_inf

bs, dim = arviz_samples.shape
new_dim = 3 * len(inf_params)
arviz_samples = arviz_samples.view(-1, len(inf_params), N)[:, :, :3].reshape(bs, new_dim) 

idata = arviz.from_dict(
    posterior={
        "params":arviz_samples.cpu().numpy()[None, :, :]
        # var: final_snapshot.cpu().numpy()[:, i] for var in var_names  # shape (chain, draw, param)
    }
) 
arviz.summary(idata)
# print(arviz.hdi(idata))
arviz.plot_posterior(idata)
plt.show()
# arviz.summary(final_snapshot.cpu().numpy(), var_names=gen_parameter_labels(inf_params, N))

# sorting in case of uniform prior

In [ ]:
# # NOTE : IF TRANSDIMENSIONAL, HAVE TO CUT OFF / MASK IRRELEVANT TOKENS FIRST, THEN SORT.
# sorted_snapshot = final_snapshot.reshape(-1, len(inf_params), N)

# # sort param columns based on peaktime row
# peaktime_idx = inf_params.index('t0')

# indeces = sorted_snapshot[:, peaktime_idx, :].argsort(dim=-1)
# indeces_expanded = indeces.unsqueeze(1).expand(-1, len(inf_params), -1)

# sorted_snapshot = torch.gather(sorted_snapshot, dim=-1, index=indeces_expanded)
# sorted_snapshot = sorted_snapshot.view(-1, vector_dim)
# sorted_snapshot

In [ ]:
# # plot sorted peaktimes
# plt.figure(figsize=(20, 3))
# indeces = np.random.choice(range(25000), 200)
# plt.plot(sorted_snapshot[indeces][:, :1].detach().cpu().numpy(), 'o')
# plt.plot(sorted_snapshot[indeces][:, 1:2].detach().cpu().numpy(), 'o')
# plt.plot(sorted_snapshot[indeces][:, 2:3].detach().cpu().numpy(), 'o')

# Classifier p(N)

In [ ]:
plt.bar(range(1, len(p_N.cpu().detach().numpy())+1), height=p_N.cpu().detach().numpy())
plt.xlabel("$N_{pred}$")
plt.title('p(N|y)')

print("Most likely estimate from classifier: ", range(1, len(p_N.cpu().detach().numpy())+1)[torch.where(p_N == torch.max(p_N))[0]])

# corner plot (for a given N)

In [ ]:
range_ = 0.97
N_inf = 6 # choose for which N to make corner plot (for FM)
N_inf = N if not transdimensional else N_inf 
# FM_samples = sorted_snapshot[(Ns == N_inf).view(Ns.size(0))]# samples where N == N_inf
FM_samples = final_snapshot[(Ns == N_inf).view(Ns.size(0))]# samples where N == N_inf
print(FM_samples.shape)
print(final_snapshot.shape)
# make vector correct size 
# (f.e. if N=2 select t0_1, t0_2, rise_1, rise_2 from vector structured like [t0_1 ... t0_Nmax,  rise_1 ... rise_Nmax])
bs, dim = FM_samples.shape
new_dim = N_inf * len(inf_params)
print( FM_samples.view(bs, len(inf_params), N)[:, :, :N_inf].shape)
FM_samples = FM_samples.view(-1, len(inf_params), N)[:, :, :N_inf].reshape(bs, new_dim)  

var_names = gen_parameter_labels(inf_params, N)
var_names = np.array(var_names).reshape(len(inf_params), N)[:, :N_inf].flatten()
fig = corner(FM_samples.cpu().numpy(), labels=var_names, range=[range_ for _ in range(N_inf * len(inf_params))], 
            color='blue', label='FM', plot_density=True, plot_datapoints=True, 
            fill_contours=False, plot_contours=True, hist_kwargs={'density':True}, bins=30, smooth=1)

plt.show()

## posterior samples

In [ ]:
# posterior samples that include all sampled component numbers

# plt.figure(figsize=(15,5))
# plt.subplot(121)
# plt.plot(observed_counts.flatten().cpu(),'k-',alpha=1,  label='noisy flux')

posterior_param_samples = path.p_simple.samples_as_dict(final_snapshot)
# Ns_10 = torch.ones_like(Ns) * 2
posterior_curve_samples = BatchedModel(
    device=device, 
    **{
        'time':torch.linspace(0, 1, observed_counts.shape[0]), 
        'burstparams':posterior_param_samples,
        # 'ybkg':3,
        'ybkg':0,
        # 'ncomp':Ns_10
        'ncomp':Ns
        }
        ).get_flux()

import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(11, 4))
gs = gridspec.GridSpec(1, 2, width_ratios=[1.5, 1])  # first subplot wider

ax1 = plt.subplot(gs[0])
ax2 = plt.subplot(gs[1])

# plt.tight_layout()
# plt.show() 

ax1.plot(np.linspace(0, 1, observed_counts.shape[0]), observed_counts.flatten().cpu(),'k-', alpha=0.7,  label='observed flux')
sample_colors=['pink', 'yellow', 'orange', 'purple', 'green', 'cyan', 'red', 'grey', 'magenta', 'blue']
for i in range(1000):
    random_idx = torch.randint(0, num_samples, size=(1,)).item()
    # ax1.plot(np.linspace(0, 1, observed_counts.shape[0]), posterior_curve_samples[random_idx].cpu(), alpha=1, color='blue', linewidth=2, label='posterior samples' if i==0 else '')
    ax1.plot(
        np.linspace(0, 1, observed_counts.shape[0]), 
        posterior_curve_samples[random_idx].cpu(), 
        alpha=1, 
        color=sample_colors[int(Ns[random_idx].item()) - 1], 
        label='posterior samples' if i==0 else ''
        )

# random_idx = torch.randint(0, num_samples, size=(1,)).item()
# ax1.plot(np.linspace(0, 1, observed_counts.shape[0]), posterior_curve_samples[random_idx].cpu(), alpha=1, color='lightgrey', label='posterior samples' if i==0 else '')

ax1.set_title(filepath.split('/')[-1].split('_')[0], fontsize=16)
# plt.ylim(top=8.5)
# ax1.legend(fontsize=12, loc='upper right')
ax1.grid(linestyle='dotted')
ax1.set_ylabel('SNR', fontsize=14)
ax1.set_xlabel('t', fontsize=14)
ax1.set_xlim(0,1)
# ax1.set_ylim(top=11)
ax1.tick_params(axis='both', labelsize=14)
# ax1.set_facecolor('lightgrey')

# plt.subplot(122)

ax2.bar(
    range(1, len(p_N.cpu().detach().numpy())+1), 
    height=p_N.cpu().detach().numpy(), 
    color=sample_colors,#'lightsteelblue', 
    edgecolor='black',    # color of the bar edge
    linewidth=1,
    label=range(1, 11)
    )
ax2.legend()
ax2.set_xlabel("$N$", fontsize=14)
ax2.set_ylabel('$p(N\mid y)$', fontsize=14, labelpad=15)
ax2.tick_params(axis='both', labelsize=14)
# ax2.set_ylim(top=0.6)
# plt.xlim(0, 11)
# plt.grid(axis='y', linestyle='dotted', zorder=-100)
import matplotlib.ticker as mticker
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()
print(observed_counts.shape)


In [ ]:
# posterior samples that include all sampled component numbers

# plt.figure(figsize=(15,5))
# plt.subplot(121)
# plt.plot(observed_counts.flatten().cpu(),'k-',alpha=1,  label='noisy flux')

posterior_param_samples = path.p_simple.samples_as_dict(final_snapshot)
# Ns_10 = torch.ones_like(Ns) * 2
posterior_curve_samples = BatchedModel(
    device=device, 
    **{
        'time':torch.linspace(0, 1, observed_counts.shape[0]), 
        'burstparams':posterior_param_samples,
        # 'ybkg':3,
        'ybkg':0,
        # 'ncomp':Ns_10
        'ncomp':Ns
        }
        ).get_flux()

import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(11, 4))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1])  # first subplot wider

ax1 = plt.subplot(gs[0])
ax2 = plt.subplot(gs[1])


ax1.plot(np.linspace(0, 1, observed_counts.shape[0]), observed_counts.flatten().cpu(),'k-', alpha=0.7,  label='observed flux')
for i in range(100):
    random_idx = torch.randint(0, num_samples, size=(1,)).item()
    ax1.plot(np.linspace(0, 1, observed_counts.shape[0]), posterior_curve_samples[random_idx].cpu(), alpha=0.2, color='red', label='posterior samples' if i==0 else '')

random_idx = torch.randint(0, num_samples, size=(1,)).item()

posterior_param_samples = path.p_simple.samples_as_dict(final_snapshot[random_idx, :])
posterior_curve_samples = BatchedModel(
    device=device, 
    **{
        'time':torch.linspace(0, 1, observed_counts.shape[0]), 
        'burstparams':posterior_param_samples,
        # 'ybkg':3,
        'ybkg':0,
        'ncomp':Ns[random_idx, :]
        }
        ).get_flux()
ax1.plot(np.linspace(0, 1, observed_counts.shape[0]), torch.mean(posterior_curve_samples.cpu(), axis=0), alpha=1, color='lightgrey', label='posterior samples' if i==0 else '')

ax1.set_title(filepath.split('/')[-1].split('_')[0], fontsize=16)
# plt.ylim(top=8.5)
# ax1.legend(fontsize=12, loc='upper right')
ax1.grid(linestyle='dotted')
ax1.set_ylabel('SNR', fontsize=14)
ax1.set_xlabel('t', fontsize=14)
ax1.set_xlim(0,1)
# ax1.set_ylim(top=11)
ax1.tick_params(axis='both', labelsize=14)
# ax1.set_facecolor('lightgrey')

# plt.subplot(122)

ax2.bar(
    range(1, len(p_N.cpu().detach().numpy())+1), 
    height=p_N.cpu().detach().numpy(), 
    color='lightsteelblue', 
    edgecolor='black', 
    linewidth=1,
    )
ax2.set_xlabel("$N$", fontsize=14)
ax2.set_ylabel('$p(N\mid y)$', fontsize=14, labelpad=15)
ax2.tick_params(axis='both', labelsize=14)
# ax2.set_ylim(top=0.6)
# plt.xlim(0, 11)
# plt.grid(axis='y', linestyle='dotted', zorder=-100)
import matplotlib.ticker as mticker
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()
print(observed_counts.shape)

In [ ]:
if transdimensional:
    MSE_loss = checkpoint['MSE_loss']
    CEL_loss = checkpoint['CEL_loss']
    plt.loglog(MSE_loss, label='MSE')
    plt.loglog(CEL_loss, label='CEL')
    plt.legend()